# Heart Disease Detection using Machine Learning

**Objective:** Predict the binary `disease` target using the supplied patient dataset.

**Workflow:** Data exploration → data quality → feature engineering → preprocessing → train/test split → multiple classifiers → evaluation → tuning/selection → feature importance → prediction.

> **Academic-use note:** This is a machine-learning demonstration, not a clinical diagnostic system.


In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
import joblib

df = pd.read_excel("Data_file.xlsx")
print("Dataset shape:", df.shape)
display(df.head())


## 1. Data exploration

The supplied dataset contains 70,000 records and 16 columns. The target variable is `disease`.

The supplied Data Information PDF describes fields including date, country, id, activity, age, alcohol, systolic/diastolic blood pressure, cholesterol, gender, glucose, height, occupation, smoking, weight and the disease target.


In [ ]:
print(df.info())
print("\nMissing values:")
display(df.isna().sum())
print("\nTarget distribution:")
display(df["disease"].value_counts())
display(df["disease"].value_counts(normalize=True))


In [ ]:
for col in ["country", "occupation", "active", "alco", "cholesterol", "gender", "gluc", "smoke", "disease"]:
    print(col, ":", sorted(df[col].dropna().unique().tolist()))


In [ ]:
# Target distribution
counts = df["disease"].value_counts().sort_index()
plt.figure(figsize=(7,5))
plt.bar(["0 - No disease", "1 - Disease"], [counts.get(0,0), counts.get(1,0)])
plt.title("Disease Class Distribution")
plt.ylabel("Patients")
plt.show()


## 2. Preprocessing and feature engineering

Features added:
- `age_years` from age in days
- `bmi` from height and weight
- `pulse_pressure` from systolic minus diastolic blood pressure

`id` is excluded because it is an identifier. `date` is explored but not used as a patient-risk predictor.

Extreme values in selected numeric variables are treated as missing and then imputed inside the ML pipeline. This avoids deleting records unnecessarily.


In [ ]:
data = df.copy()
data["age_years"] = data["age"] / 365.25
data["bmi"] = data["weight"] / ((data["height"]/100)**2)
data["pulse_pressure"] = data["ap_hi"] - data["ap_lo"]

bounds = {
    "age_years": (18,100), "ap_hi": (70,250), "ap_lo": (40,200),
    "height": (120,220), "weight": (30,200), "bmi": (10,70),
    "pulse_pressure": (0,200)
}
for col,(lo,hi) in bounds.items():
    data.loc[(data[col] < lo) | (data[col] > hi), col] = np.nan

FEATURES = ["country","active","age_years","alco","ap_hi","ap_lo","cholesterol","gender",
            "gluc","height","occupation","smoke","weight","bmi","pulse_pressure"]
CAT_FEATURES = ["country","occupation"]
NUM_FEATURES = [c for c in FEATURES if c not in CAT_FEATURES]

X = data[FEATURES]
y = data["disease"].astype(int)

X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=0.20,stratify=y,random_state=42
)

def make_preprocessor():
    return ColumnTransformer([
        ("num", Pipeline([
            ("imputer",SimpleImputer(strategy="median")),
            ("scaler",StandardScaler())
        ]), NUM_FEATURES),
        ("cat", Pipeline([
            ("imputer",SimpleImputer(strategy="most_frequent")),
            ("onehot",OneHotEncoder(handle_unknown="ignore"))
        ]), CAT_FEATURES)
    ])


## 3. Train and compare classification models

Models:
- Logistic Regression
- Decision Tree
- Random Forest
- Linear SVM

The comparison uses Accuracy, Precision, Recall, F1-score and ROC-AUC.


In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Decision Tree": DecisionTreeClassifier(max_depth=6, min_samples_leaf=20, class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=300, max_depth=14, min_samples_leaf=5, class_weight="balanced_subsample", n_jobs=-1, random_state=42),
    "Linear SVM": LinearSVC(C=1.0, class_weight="balanced", max_iter=5000, random_state=42)
}

results = []
fitted = {}

for name, estimator in models.items():
    pipe = Pipeline([("preprocessor",make_preprocessor()),("model",estimator)])
    pipe.fit(X_train,y_train)
    pred = pipe.predict(X_test)
    score = pipe.predict_proba(X_test)[:,1] if hasattr(pipe,"predict_proba") else pipe.decision_function(X_test)
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test,pred),
        "Precision": precision_score(y_test,pred,zero_division=0),
        "Recall": recall_score(y_test,pred,zero_division=0),
        "F1-Score": f1_score(y_test,pred,zero_division=0),
        "ROC-AUC": roc_auc_score(y_test,score)
    })
    fitted[name] = pipe

results_df = pd.DataFrame(results).sort_values("F1-Score",ascending=False)
display(results_df)


In [ ]:
# Comparison chart
plot_df = results_df.set_index("Model")[["Accuracy","Precision","Recall","F1-Score"]]
plot_df.plot(kind="bar",figsize=(10,5),ylim=(0,1))
plt.title("Model Performance Comparison")
plt.ylabel("Score")
plt.xticks(rotation=25,ha="right")
plt.tight_layout()
plt.show()


## 4. Final model

Random Forest is selected for the presentation model because it provides strong overall performance and supports feature-importance analysis. The model is trained on the training split and evaluated on the untouched test split.


In [ ]:
final_model = fitted["Random Forest"]
final_pred = final_model.predict(X_test)
final_prob = final_model.predict_proba(X_test)[:,1]

print(classification_report(y_test,final_pred,digits=4))
print("Accuracy :",accuracy_score(y_test,final_pred))
print("Precision:",precision_score(y_test,final_pred))
print("Recall   :",recall_score(y_test,final_pred))
print("F1-score :",f1_score(y_test,final_pred))
print("ROC-AUC  :",roc_auc_score(y_test,final_prob))


In [ ]:
cm = confusion_matrix(y_test,final_pred)
plt.figure(figsize=(6,5))
plt.imshow(cm,interpolation="nearest")
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.xticks([0,1],["0","1"]); plt.yticks([0,1],["0","1"])
for i in range(2):
    for j in range(2):
        plt.text(j,i,cm[i,j],ha="center",va="center")
plt.colorbar()
plt.show()


In [ ]:
# Feature importance
feature_names = final_model.named_steps["preprocessor"].get_feature_names_out()
importance = final_model.named_steps["model"].feature_importances_
importance_df = pd.DataFrame({"Feature":feature_names,"Importance":importance}).sort_values("Importance",ascending=False)
display(importance_df.head(15))

top = importance_df.head(15).sort_values("Importance")
plt.figure(figsize=(9,6))
plt.barh(top["Feature"],top["Importance"])
plt.title("Top Feature Importances")
plt.xlabel("Importance")
plt.show()


In [ ]:
joblib.dump(final_model,"best_model.joblib")
print("Final model saved as best_model.joblib")


## 5. Insights and conclusion

In this dataset, the strongest model-based predictors are primarily blood-pressure-related variables, age and measures related to cholesterol/body composition.

These are **predictive patterns in the dataset**, not proof of causation. For a health-risk application, recall should be monitored carefully because false negatives can be important.

The Streamlit interface in `app.py` uses the same saved preprocessing pipeline and model, so the presentation UI is consistent with the notebook.
